# NavState IMU: PIM vs EKF Covariance and NEES

This notebook compares the covariance and NEES produced by:

- `NavStateImuEKF` (predict only, no measurement updates).
- `PreintegratedImuMeasurements` (default preintegration type, often tangent).
- `PreintegratedImuMeasurementsManifold` (manifold preintegration).

We run a Monte Carlo simulation with synthetic IMU data and check that the two
preintegration strategies yield the same covariance and NEES, and that they
match the EKF predict step.


In [1]:
# Install GTSAM from pip if running in Google Colab
try:
    import google.colab  # type: ignore
    %pip install --quiet gtsam-develop
except Exception:
    pass  # Not in Colab

import numpy as np
import plotly.graph_objects as go

import gtsam
from gtsam import ConstantTwistScenario, ScenarioRunner

### NEES definition

The Normalized Estimation Error Squared (NEES) is

$$\mathrm{NEES} = e^T P^{-1} e,$$

where $e$ is the 9D NavState error (local coordinates) and $P$ is the predicted
covariance. A consistent filter should have mean NEES near the state dimension (9).


In [2]:
from dataclasses import dataclass


@dataclass
class ErrorCov:
    error: np.ndarray
    covariance: np.ndarray


@dataclass
class TrialResult:
    default: ErrorCov
    manifold: ErrorCov
    ekf: ErrorCov


def nees(error, covariance):
    """Compute NEES = e^T P^{-1} e."""
    return float(error.T @ np.linalg.solve(covariance, error))


def rel_frobenius(A, B):
    """Relative Frobenius norm ||A-B|| / ||B||."""
    return float(np.linalg.norm(A - B, ord="fro") / np.linalg.norm(B, ord="fro"))


def delta_error(delta_est, delta_true, exact):
    """Delta-space error: exact uses group difference of deltas."""
    if exact:
        return gtsam.NavState.Expmap(delta_est).localCoordinates(
            gtsam.NavState.Expmap(delta_true)
        )
    return delta_est - delta_true


def simulate_interval(interval_s, num_trials, dt, params, scenario, P0, seed=0):
    """Monte Carlo for a single preintegration interval.

    Returns per-trial errors and predicted covariances for:
    - default PIM
    - manifold PIM
    - NavStateImuEKF
    """
    steps = int(round(interval_s / dt))
    if abs(steps * dt - interval_s) > 1e-12:
        raise ValueError("interval_s must be an integer multiple of dt")

    rng = np.random.default_rng(seed)
    bias = gtsam.imuBias.ConstantBias(np.zeros(3), np.zeros(3))
    runner = ScenarioRunner(scenario, params, dt, bias)

    X0_nominal = scenario.navState(0.0)
    X_true = scenario.navState(interval_s)
    delta_true = X0_nominal.localCoordinates(X_true)

    trials = []

    for _ in range(num_trials):
        xi0 = rng.multivariate_normal(np.zeros(9), P0)
        X0 = X0_nominal.retract(xi0)

        pim_default = gtsam.PreintegratedImuMeasurements(params, bias)
        pim_manifold = gtsam.PreintegratedImuMeasurementsManifold(params, bias)
        ekf = gtsam.NavStateImuEKF(X0, P0, params)

        for k in range(steps):
            t = k * dt
            omega_meas = runner.measuredAngularVelocity(t)
            acc_meas = runner.measuredSpecificForce(t)

            pim_default.integrateMeasurement(acc_meas, omega_meas, dt)
            pim_manifold.integrateMeasurement(acc_meas, omega_meas, dt)
            ekf.predict(omega_meas, acc_meas, dt)

        X_pred_default = pim_default.predict(X0, bias)
        X_pred_manifold = pim_manifold.predict(X0, bias)
        X_pred_ekf = ekf.state()

        trial = TrialResult(
            default=ErrorCov(
                error=delta_error(X0.localCoordinates(X_pred_default), delta_true, True),
                covariance=P0 + pim_default.preintMeasCov(),
            ),
            manifold=ErrorCov(
                error=delta_error(X0.localCoordinates(X_pred_manifold), delta_true, True),
                covariance=P0 + pim_manifold.preintMeasCov(),
            ),
            ekf=ErrorCov(
                error=X_true.localCoordinates(X_pred_ekf), covariance=ekf.covariance()
            ),
        )

        trials.append(trial)

    return {
        "trials": trials,
    }

### Scenario and IMU parameters

We use a constant yaw-rate scenario with steady velocity. The EKF only runs
predict, and the PIMs only integrate IMU measurements.


### Initial uncertainty

We sample an initial NavState by drawing a 9D delta from `P0` and applying `retract`,
which is equivalent to using `NavState::Expmap` at the current state.
We transport `P0` through the PIM prediction Jacobian `H1` so the
PIM covariance is expressed in the NavState tangent space before NEES.
For PIM NEES we compute the error in **delta space anchored at X_i** using either
the exact group error or a simple delta subtraction (controlled by `use_exact_delta_error`).


In [3]:
# Scenario: steady yaw rate with constant world-frame velocity
radius = 30.0
angular_velocity = np.pi  # rad/sec
w_b = np.array([0.0, 0.0, angular_velocity])
v_n = np.array([radius * angular_velocity, 0.0, 0.0])
scenario = ConstantTwistScenario(w_b, v_n)

# IMU noise params (NavState, Z-up)
params = gtsam.PreintegrationParams.MakeSharedU(9.81)
accel_noise_sigma = 0.1
gyro_noise_sigma = 0.01
params.setAccelerometerCovariance(np.eye(3) * accel_noise_sigma**2)
params.setGyroscopeCovariance(np.eye(3) * gyro_noise_sigma**2)
params.setIntegrationCovariance(np.eye(3) * 1e-8)

# Monte Carlo configuration
dt = 0.01  # 100 Hz IMU
intervals = np.array([50, 100, 150, 200]) / 1000  # milliseconds
num_trials = 200

# Initial covariance (NavState tangent order: w, p, v)
init_sigmas = np.array([0.02, 0.02, 0.02, 0.2, 0.2, 0.2, 0.5, 0.5, 0.5])
P0 = np.diag(init_sigmas**2)

use_exact_delta_error = (
    True  # True: group error in delta space, False: delta subtraction
)

### Run Monte Carlo across multiple preintegration intervals


In [4]:
results = []
for interval_s in intervals:
    data = simulate_interval(interval_s, num_trials, dt, params, scenario, P0, seed=int(1000 * interval_s))
    trials = data["trials"]

    nees_default = np.array([nees(t.default.error, t.default.covariance) for t in trials])
    nees_manifold = np.array([nees(t.manifold.error, t.manifold.covariance) for t in trials])
    nees_ekf = np.array([nees(t.ekf.error, t.ekf.covariance) for t in trials])

    cov_mean_default = np.mean([t.default.covariance for t in trials], axis=0)
    cov_mean_manifold = np.mean([t.manifold.covariance for t in trials], axis=0)
    cov_mean_ekf = np.mean([t.ekf.covariance for t in trials], axis=0)

    emp_cov = np.cov(np.stack([t.default.error for t in trials], axis=0).T, bias=True)

    results.append({
        "interval_s": interval_s,
        "nees_default_mean": float(nees_default.mean()),
        "nees_default_std": float(nees_default.std(ddof=0)),
        "nees_manifold_mean": float(nees_manifold.mean()),
        "nees_manifold_std": float(nees_manifold.std(ddof=0)),
        "nees_ekf_mean": float(nees_ekf.mean()),
        "nees_ekf_std": float(nees_ekf.std(ddof=0)),
        "rel_cov_default_vs_manifold": rel_frobenius(cov_mean_default, cov_mean_manifold),
        "rel_cov_default_vs_ekf": rel_frobenius(cov_mean_default, cov_mean_ekf),
        "rel_cov_manifold_vs_ekf": rel_frobenius(cov_mean_manifold, cov_mean_ekf),
        "rel_cov_default_vs_emp": rel_frobenius(cov_mean_default, emp_cov),
        "rel_cov_ekf_vs_emp": rel_frobenius(cov_mean_ekf, emp_cov),
    })

results


[{'interval_s': np.float64(0.05),
  'nees_default_mean': 0.732541283415913,
  'nees_default_std': 0.45064446831420696,
  'nees_manifold_mean': 0.7325417437759313,
  'nees_manifold_std': 0.4506453732727786,
  'nees_ekf_mean': 8.578577369813082,
  'nees_ekf_std': 3.9847842910628835,
  'rel_cov_default_vs_manifold': 0.0001334517326373641,
  'rel_cov_default_vs_ekf': 0.23647551460143135,
  'rel_cov_manifold_vs_ekf': 0.23646635627849075,
  'rel_cov_default_vs_emp': 33.32910001244249,
  'rel_cov_ekf_vs_emp': 41.45994771378634},
 {'interval_s': np.float64(0.1),
  'nees_default_mean': 2.787998817033879,
  'nees_default_std': 1.7413000172544384,
  'nees_manifold_mean': 2.7878594759967203,
  'nees_manifold_std': 1.7412603227374222,
  'nees_ekf_mean': 8.81305571909107,
  'nees_ekf_std': 3.855761694968445,
  'rel_cov_default_vs_manifold': 0.0024585853420875346,
  'rel_cov_default_vs_ekf': 0.5603329657944346,
  'rel_cov_manifold_vs_ekf': 0.5601135667787372,
  'rel_cov_default_vs_emp': 8.33402308404

### Numerical summary


In [5]:
def print_table(rows):
    headers = [
        "interval_s",
        "nees_default_mean",
        "nees_manifold_mean",
        "nees_ekf_mean",
        "rel_cov_default_vs_manifold",
        "rel_cov_default_vs_ekf",
    ]
    widths = {h: max(len(h), 14) for h in headers}
    for row in rows:
        for h in headers:
            widths[h] = max(widths[h], len(f"{row[h]:.4f}") if isinstance(row[h], float) else len(str(row[h])))

    header_line = " | ".join(f"{h:<{widths[h]}}" for h in headers)
    print(header_line)
    print("-" * len(header_line))

    for row in rows:
        line_parts = []
        for h in headers:
            val = row[h]
            if isinstance(val, float):
                line_parts.append(f"{val:<{widths[h]}.4f}")
            else:
                line_parts.append(f"{val:<{widths[h]}}")
        print(" | ".join(line_parts))

print_table(results)


interval_s     | nees_default_mean | nees_manifold_mean | nees_ekf_mean  | rel_cov_default_vs_manifold | rel_cov_default_vs_ekf
-------------------------------------------------------------------------------------------------------------------------------
0.0500         | 0.7325            | 0.7325             | 8.5786         | 0.0001                      | 0.2365                
0.1000         | 2.7880            | 2.7879             | 8.8131         | 0.0025                      | 0.5603                
0.1500         | 6.6820            | 6.6801             | 8.9279         | 0.0125                      | 0.7433                
0.2000         | 10.7810           | 10.7596            | 8.7755         | 0.0378                      | 0.8344                


### NEES vs interval


In [6]:
dim = 9
interval_ms = [int(1000 * r["interval_s"]) for r in results]

fig = go.Figure()
fig.add_scatter(
    x=interval_ms,
    y=[r["nees_default_mean"] for r in results],
    mode="lines+markers",
    name="PIM default",
)
fig.add_scatter(
    x=interval_ms,
    y=[r["nees_manifold_mean"] for r in results],
    mode="lines+markers",
    name="PIM manifold",
)
fig.add_scatter(
    x=interval_ms,
    y=[r["nees_ekf_mean"] for r in results],
    mode="lines+markers",
    name="NavStateImuEKF",
)
fig.add_scatter(
    x=interval_ms,
    y=[dim] * len(results),
    mode="lines",
    name="Expected NEES (dim)",
    line=dict(color="black", dash="dash"),
)
fig.update_layout(
    title="Mean NEES vs preintegration interval",
    xaxis_title="Interval (ms)",
    yaxis_title="Mean NEES",
)
fig.show()


### Covariance consistency metrics


In [7]:
fig = go.Figure()
fig.add_scatter(
    x=interval_ms,
    y=[r["rel_cov_default_vs_manifold"] for r in results],
    mode="lines+markers",
    name="Default vs Manifold",
)
fig.add_scatter(
    x=interval_ms,
    y=[r["rel_cov_default_vs_ekf"] for r in results],
    mode="lines+markers",
    name="Default vs EKF",
)
fig.add_scatter(
    x=interval_ms,
    y=[r["rel_cov_manifold_vs_ekf"] for r in results],
    mode="lines+markers",
    name="Manifold vs EKF",
)
fig.update_layout(
    title="Relative covariance differences",
    xaxis_title="Interval (ms)",
    yaxis_title="Relative Frobenius norm",
)
fig.show()

## Interpretation

- The NEES curves should sit near 9 if the covariance is consistent.
- The two PIM implementations should produce nearly identical covariance and NEES.
- The EKF predict covariance should match the PIM covariance for the same IMU data.
